# ASR quality — per-language overview + WER as a quality signal

Sections:
1. Split distribution (validated / other / invalidated)
2. Aggregate WER and CER per language
3. Up/down vote pair distribution from SHAR `cut.custom`
4. Per-cut WER/CER stratified by split (incl. validated, with semantic colors)
5. Per-cut WER/CER vs upvotes alone, downvotes alone
6. **The (up, down) pair as a unit**: 2D heatmap, net_score boxplots
7. **Quality-signal validation**: Spearman correlations + ROC AUC
8. Operating-point curves (precision/recall vs WER threshold)
9. Audio inspector slider — drag through WER values, hear samples

Sections 6-8 directly answer: *can I trust WER as a quality proxy on a dataset that has no human votes?* CommonVoice gives us the labelled ground truth to calibrate that confidence.

In [ ]:
import sys
from pathlib import Path

_HELPER_DIR = Path('..').resolve()
if str(_HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(_HELPER_DIR))

from asr_quality_helper import (
    load_asr_dataframe,
    wer_cer_per_language,
    add_per_cut_wer_cer,
    add_vote_features,
    detect_vote_columns,
    plot_split_distribution,
    plot_wer_cer,
    plot_vote_distribution,
    plot_metric_distribution_by_split,
    plot_metric_vs_votes,
    plot_metric_heatmap_by_vote_pair,
    plot_metric_vs_net_score,
    compute_quality_correlations,
    plot_quality_roc,
    plot_quality_threshold_curves,
    build_cut_lookup,
    metric_slider_inspector,
)

import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

In [ ]:
# Paths — adjust per dataset.
ASR_ROOT  = Path('/capstor/scratch/cscs/sgodey/quality_assesment_output/results/asr/commonvoice22_sidon')
SHAR_ROOT = Path('/capstor/scratch/cscs/sgodey/audio-datasets/SHAR/stage_2/commonvoice22_sidon')

WORKERS = 8
LANG_FILTER = None  # None = all langs (recommended); set ['de'] etc for fast iteration

## 1. Load the joined DataFrame

In [ ]:
df = load_asr_dataframe(ASR_ROOT, SHAR_ROOT, workers=WORKERS, lang_filter=LANG_FILTER)
print(f'{len(df):,} rows | langs={sorted(df["lang"].unique())} | splits={sorted(df["split"].unique())}')
df.head(3)

## 2. Split distribution per language

Color scheme used everywhere downstream: validated/validation = green, other = blue, invalidated = red.

In [ ]:
plot_split_distribution(df)
plt.show()
df.groupby(['lang', 'split']).size().unstack(fill_value=0)

## 3. Aggregate WER/CER per language

In [ ]:
stats = wer_cer_per_language(df)
stats

In [ ]:
plot_wer_cer(stats); plt.show()
plot_vote_distribution(df); plt.show()

## 4. Per-cut WER/CER. Compute and add vote-derived features.

In [ ]:
df = add_per_cut_wer_cer(df, workers=8)
df = add_vote_features(df)
df[['lang', 'split', 'wer', 'cer'] + [c for c in ['up_votes','down_votes','net_score','vote_total','vote_ratio'] if c in df.columns]].head()

### 4a. WER distribution by split (per language) — validated stands out in green

In [ ]:
plot_metric_distribution_by_split(df, metric='wer', bins=50, clip_max=1.5)
plt.show()

### 4b. CER distribution by split

In [ ]:
plot_metric_distribution_by_split(df, metric='cer', bins=50, clip_max=1.0)
plt.show()

## 5. WER vs upvotes / downvotes individually (for reference)

In [ ]:
up_col, down_col = detect_vote_columns(df)
plot_metric_vs_votes(df, metric='wer', vote_col=up_col); plt.show()
plot_metric_vs_votes(df, metric='wer', vote_col=down_col); plt.show()

## 6. The (up, down) pair as a single unit

What actually carries human judgment is the *pair* together — `(3↑, 0↓)` and `(0↑, 3↓)` mean very different things even though both have `vote_total=3`. The next two views collapse the pair appropriately:

- **6a.** 2D heatmap of mean WER per (up, down) cell. Cells with < 5 samples are masked. Same color scale across panels so languages compare visually.
- **6b.** Boxplot of WER vs `net_score = up - down`, the simplest scalar quality axis from the pair.

In [ ]:
plot_metric_heatmap_by_vote_pair(df, metric='wer', max_v=6, min_cell=5)
plt.show()

In [ ]:
plot_metric_vs_net_score(df, metric='wer', clip=5)
plt.show()

## 7. WER as a quality signal — correlations + ROC

**The headline plot for your stated goal.** Once you take WER to a dataset *without* human votes, you'll need a confidence number for "is WER actually informative here?". Two pieces:

- **7a.** Spearman correlations between WER and each vote-derived signal (sign-flipped so positive = expected direction). High correlation with `net_score` and `vote_ratio` means "WER tracks the human vote pair".
- **7b.** ROC of `-WER` discriminating *validation* vs *invalidated*. AUC = the probability that a randomly-chosen validated cut has lower WER than a randomly-chosen invalidated one. AUC ≈ 1.0 means "WER alone recovers the human good/bad label perfectly". This is the number you cite when applying WER on new data.

In [ ]:
corr = compute_quality_correlations(df, metric='wer')
print('Spearman ρ between -WER and each quality signal (positive = expected sign):')
corr

In [ ]:
aucs, _ = plot_quality_roc(df, metric='wer',
                            positive_split='validation',
                            negative_split='invalidated')
plt.show()
print('Per-language AUC of (-WER) for predicting validation vs invalidated:')
for lang, auc in sorted(aucs.items()):
    print(f'  {lang}: {auc:.3f}')

## 8. Threshold operating points

If you decide to filter by WER on a new dataset, what threshold makes sense? These curves are calibrated against CommonVoice's human labels:

- **green** — precision: of cuts I keep at threshold T, what fraction are actually validated?
- **blue** — recall: of all validated cuts, what fraction do I keep?
- **gray** — fraction of *all* cuts kept (sanity check on dataset retention)

Pick a T where the green precision is high enough for your downstream pipeline, then read off how much data you'll keep.

In [ ]:
plot_quality_threshold_curves(df, metric='wer',
                                positive_split='validation',
                                negative_split='invalidated')
plt.show()

### Optional — vote-pair count heatmap (counts only, not WER)

In [ ]:
if up_col and down_col:
    import numpy as np
    sub = df.copy()
    sub[up_col] = pd.to_numeric(sub[up_col], errors='coerce').fillna(0).astype(int)
    sub[down_col] = pd.to_numeric(sub[down_col], errors='coerce').fillna(0).astype(int)
    langs = sorted(sub['lang'].unique())
    fig, axes = plt.subplots(1, len(langs), figsize=(4*len(langs), 4), sharey=True)
    if len(langs) == 1:
        axes = [axes]
    for ax, lang in zip(axes, langs):
        g = sub[sub['lang']==lang]
        h = g.groupby([down_col, up_col]).size().unstack(fill_value=0)
        im = ax.imshow(h.values, origin='lower', aspect='auto', cmap='viridis')
        ax.set_title(f'{lang}  (n={len(g):,})')
        ax.set_xlabel(up_col)
        ax.set_ylabel(down_col if ax is axes[0] else '')
        ax.set_xticks(range(len(h.columns))); ax.set_xticklabels(h.columns, fontsize=8)
        ax.set_yticks(range(len(h.index))); ax.set_yticklabels(h.index, fontsize=8)
    fig.suptitle('vote-pair counts per language (down × up)', y=1.02)
    plt.tight_layout(); plt.show()

## 9. Audio inspector — slider over WER (or CER)

**Requires lhotse + soundfile installed in your kernel's venv.** I added them to `~/.venv-tools` (just stdlib lhotse, no torch). If your kernel uses a different venv (e.g. the container's `/opt/venv`), they're already there.

Caches the cut_lookup to a pickle so kernel restarts skip the rebuild. Pass `keep_ids=set(df['cut_id'])` to keep the lookup small.

**Tip:** restrict `LANG_FILTER` (cell 2) to a single language before this section — building cut_lookup for all 7 langs takes a few minutes.

In [ ]:
CACHE = Path('/capstor/scratch/cscs/sgodey/quality_assesment_output/caches/notebook/asr_quality_cut_lookup_de.pkl')
cut_lookup = build_cut_lookup(
    SHAR_ROOT,
    keep_ids=set(df['cut_id']),
    languages=LANG_FILTER,
    cache_path=CACHE,
)
print(f'cut_lookup size: {len(cut_lookup):,}')

In [ ]:
metric_slider_inspector(
    df,
    cut_lookup,
    metric='wer',     # or 'cer'
    n_samples=3,
    n_bins=50,
    max_seconds=10.0,
    seed=None,
)